<a href="https://colab.research.google.com/github/Suhani0411/OOD-Detection/blob/main/Baseline_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

from PIL import ImageFile

ImageFile.LOAD_TRUNCATED_IMAGES = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!unzip -q "/content/drive/MyDrive/dataset/archive.zip" -d "/content/dataset"

print("Cat-Dog dataset extracted!")

Cat-Dog dataset extracted!


In [ ]:
!unzip -q "/content/drive/MyDrive/ood/ood.zip" -d "/content/ood_images"

print("OOD dataset extracted!")

OOD dataset extracted!


In [ ]:
import os

print(os.listdir("/content/ood_images/dataset")[:20])
print("Count:", len(os.listdir("/content/ood_images/dataset")))

['baby-african-elephant.jpeg', 'scissor.jpeg', '41QKCkQ2A5L._AC_UF894,1000_QL80_.jpg', 'bottle.jpg', '61FYlYtnZCL._AC_UF1000,1000_QL80_.jpg', 'sofa.jpg', '7152QM01_1.jpeg', 'eraser.jpg', '.DS_Store', 'phone.jpg', 'hq720.jpg', 'lamp.jpg', '013025_ED_giraffe-zebra_feat.jpg', '308074031-480px.jpg', 'Cow_(Fleckvieh_breed)_Oeschinensee_Slaunger_2009-07-07.jpg', 'fan.jpg', 'clock.jpg', '615xjNdE7+L._AC_UF894,1000_QL80_.jpg', 'spoon.jpg', 'Understanding-the-Basics-of-Golf-Clubs.jpg']
Count: 100


In [ ]:
import os

print(os.listdir("/content/dataset"))

['PetImages']


In [ ]:
train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


val_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [ ]:
full_dataset = datasets.ImageFolder(
    root="/content/dataset/PetImages"
)

In [ ]:
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(
    full_dataset,
    [train_size, val_size]
)

In [ ]:
train_dataset.dataset.transform = train_transform

val_dataset.dataset.transform = val_transform

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

In [ ]:
class GaussianLayer(nn.Module):

    def __init__(self, in_features, num_classes):
        super(GaussianLayer, self).__init__()

        self.num_classes = num_classes
        self.in_features = in_features


        # Learnable class centers
        self.mu = nn.Parameter(
            torch.randn(num_classes, in_features)
        )


        # Learnable spreads
        self.sigma = nn.Parameter(
            torch.ones(num_classes, in_features)
        )


    def forward(self, x):

        # x shape:
        # [batch_size, in_features]

        x = x.unsqueeze(1)

        mu = self.mu.unsqueeze(0)

        sigma = torch.abs(self.sigma.unsqueeze(0)) + 1e-6


        # Mahalanobis-like distance
        distances = ((x - mu) ** 2) / (2 * sigma ** 2)


        # Negative distance as logits
        logits = -distances.sum(dim=2)


        return logits, distances.sum(dim=2)

In [ ]:
model = models.efficientnet_b0(pretrained=True)

num_features = model.classifier[1].in_features

model.classifier[1] = GaussianLayer(
    num_features,
    2
)

model = model.to(device)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 143MB/s]


In [ ]:
for param in model.features.parameters():
    param.requires_grad = False

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

#scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    #optimizer,
    #mode='min',
    #factor=0.5,
    #patience=1
#)

In [ ]:
scaler = torch.cuda.amp.GradScaler()

/tmp/ipykernel_5064/2340218076.py:1: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0

    correct = 0
    total = 0


    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)


        optimizer.zero_grad()


        outputs,distances = model(images)

        loss = criterion(outputs, labels)


        loss.backward()

        optimizer.step()


        running_loss += loss.item()


        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()


    accuracy = 100 * correct / total


    print(f"\nEpoch [{epoch+1}/{num_epochs}]")

    print(f"Loss: {running_loss/len(train_loader):.4f}")

    print(f"Train Accuracy: {accuracy:.2f}%")


Epoch [1/10]
Loss: 3.5187
Train Accuracy: 72.07%

Epoch [2/10]
Loss: 1.3686
Train Accuracy: 86.28%

Epoch [3/10]
Loss: 1.0493
Train Accuracy: 89.51%

Epoch [4/10]
Loss: 0.9778
Train Accuracy: 90.66%

Epoch [5/10]
Loss: 0.8826
Train Accuracy: 91.20%

Epoch [6/10]
Loss: 0.8707
Train Accuracy: 91.45%

Epoch [7/10]
Loss: 0.7769
Train Accuracy: 92.15%

Epoch [8/10]
Loss: 0.7526
Train Accuracy: 92.17%

Epoch [9/10]
Loss: 0.6790
Train Accuracy: 92.83%

Epoch [10/10]
Loss: 0.6658
Train Accuracy: 92.62%


In [ ]:
import torch.nn.functional as F

def get_confidence(model, image_tensor):

    model.eval()

    with torch.no_grad():

        output,distances = model(image_tensor)
        min_distance=distances.min().item();

        probabilities = F.softmax(output, dim=1)

        confidence, prediction = torch.max(probabilities, 1)


    return (
        prediction.item(),
        confidence.item(),
        probabilities.cpu().numpy()
    )

In [ ]:
#for param in model.features.parameters():
    #param.requires_grad = True

In [ ]:
#optimizer = optim.Adam(
    #model.parameters(),
    #lr=0.0001
#)

In [ ]:
#fine_tune_epochs = 5

#for epoch in range(fine_tune_epochs):

    #model.train()

    #correct_train = 0
   # total_train = 0


    #for images, labels in train_loader:

        #images = images.to(device)
        #labels = labels.to(device)


        #optimizer.zero_grad()


        #with torch.cuda.amp.autocast():

            #outputs = model(images)

            #loss = criterion(outputs, labels)


        #scaler.scale(loss).backward()

        #scaler.step(optimizer)

       # scaler.update()


       # _, predicted = torch.max(outputs, 1)

       # total_train += labels.size(0)

       # correct_train += (predicted == labels).sum().item()


    #train_accuracy = 100 * correct_train / total_train


    # VALIDATION
   # model.eval()

    #correct_val = 0
   # total_val = 0


   # with torch.no_grad():

       # for images, labels in val_loader:

           # images = images.to(device)
           # labels = labels.to(device)


          #  outputs = model(images)

          #  _, predicted = torch.max(outputs, 1)
          #

          #  total_val += labels.size(0)
          #
          #  correct_val += (predicted == labels).sum().item()


    #val_accuracy = 100 * correct_val / total_val


    #if val_accuracy > best_accuracy:

      #  best_accuracy = val_accuracy

      #  torch.save(model.state_dict(), "best_model.pth")


   # print(f"\nFine-Tune Epoch [{epoch+1}/{fine_tune_epochs}]")

   # print(f"Train Accuracy: {train_accuracy:.2f}%")

    #print(f"Validation Accuracy: {val_accuracy:.2f}%")
   #
   # print(f"Best Accuracy: {best_accuracy:.2f}%")

In [ ]:
#model.load_state_dict(torch.load("best_model.pth"))

#model.eval()

In [ ]:
#from google.colab import files

#uploaded = files.upload()

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import torch.nn.functional as F
import os

class_names = ['Cat', 'Dog']

transform_test = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

model.eval()

THRESHOLD = 0.7

image_folder = "/content/ood_images/dataset"
image_files = [
    f for f in os.listdir(image_folder)
    if f.lower().endswith(('.jpg', '.jpeg', '.png', '.jfif'))
]

ood_count = 0

for image_name in image_files:

    image_path = os.path.join(image_folder, image_name)

    image = Image.open(image_path).convert('RGB')

    image_tensor = transform_test(image)

    image_tensor = image_tensor.unsqueeze(0)

    image_tensor = image_tensor.to(device)

    with torch.no_grad():

        logits, distances = model(image_tensor)

        probabilities = F.softmax(logits, dim=1)

        confidence, prediction = torch.max(probabilities, 1)

        confidence_score = confidence.item()

    if confidence_score < THRESHOLD:

        print(f"{image_name} --> Unknown (OOD Detected) | Confidence: {confidence_score*100:.2f}%")
        ood_count += 1

    else:

        predicted_class = class_names[prediction.item()]

        print(f"{image_name} --> {predicted_class} | Confidence: {confidence_score*100:.2f}%")

print("-" * 60)
print(f"Detected {ood_count} out of {len(image_files)} images as OOD")
print(f"OOD Detection Rate: {(ood_count/len(image_files))*100:.2f}%")

baby-african-elephant.jpeg --> Dog | Confidence: 100.00%
scissor.jpeg --> Dog | Confidence: 100.00%
41QKCkQ2A5L._AC_UF894,1000_QL80_.jpg --> Dog | Confidence: 100.00%
bottle.jpg --> Dog | Confidence: 100.00%
61FYlYtnZCL._AC_UF1000,1000_QL80_.jpg --> Dog | Confidence: 99.98%
sofa.jpg --> Unknown (OOD Detected) | Confidence: 64.57%
7152QM01_1.jpeg --> Dog | Confidence: 100.00%
eraser.jpg --> Dog | Confidence: 82.57%
phone.jpg --> Unknown (OOD Detected) | Confidence: 56.70%
hq720.jpg --> Dog | Confidence: 100.00%
lamp.jpg --> Dog | Confidence: 99.96%
013025_ED_giraffe-zebra_feat.jpg --> Dog | Confidence: 100.00%
308074031-480px.jpg --> Dog | Confidence: 100.00%
Cow_(Fleckvieh_breed)_Oeschinensee_Slaunger_2009-07-07.jpg --> Dog | Confidence: 100.00%
fan.jpg --> Dog | Confidence: 99.93%
clock.jpg --> Dog | Confidence: 100.00%
615xjNdE7+L._AC_UF894,1000_QL80_.jpg --> Dog | Confidence: 99.99%
spoon.jpg --> Dog | Confidence: 99.99%
Understanding-the-Basics-of-Golf-Clubs.jpg --> Dog | Confidenc

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


bed.jpg.jpg --> Cat | Confidence: 99.94%
ref.jpg --> Dog | Confidence: 99.90%
download.jpeg --> Dog | Confidence: 99.99%
images (9).jpeg --> Dog | Confidence: 100.00%
kettle.jpg --> Dog | Confidence: 87.84%
chair.jpeg --> Dog | Confidence: 100.00%
dairy-goats-1_2x_49a19023-b661-41ef-8796-8e044e5dfb46.jpeg --> Dog | Confidence: 100.00%
photo.jpeg --> Dog | Confidence: 97.87%
helicopter.jpg --> Dog | Confidence: 100.00%
racquet_and_ball.jpg --> Dog | Confidence: 100.00%
tv.jpg --> Dog | Confidence: 100.00%
camera.jpeg --> Dog | Confidence: 100.00%
how-to-draw-crossed-hockey-sticks-featured-image-1200.png --> Cat | Confidence: 99.99%
61pFab9tNeL.jpg --> Dog | Confidence: 99.97%
headphones.jpg --> Dog | Confidence: 100.00%
train.jpg --> Dog | Confidence: 100.00%
blender.jpg --> Dog | Confidence: 70.14%
airplane.jpg.jpg --> Dog | Confidence: 99.84%
calculator.jpg --> Dog | Confidence: 99.94%
mirror.jpg --> Dog | Confidence: 99.20%
Mango_6fb74c95-c9b0-4559-88e8-f542e6d6b18d.jpeg --> Dog | Co